# Structural Vision AR — Model 1: Crack Segmentation
**Before running:** Runtime > Change runtime type > T4 GPU

In [ ]:
# 1. Mount Drive — your model saves here, survives disconnects
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/structural_vision/model1'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# 2. Upload crack_dataset.zip when prompted
from google.colab import files
uploaded = files.upload()   # upload crack_dataset.zip here

In [ ]:
# 3. Extract + install
!unzip -q crack_dataset.zip -d /content/datasets/
!pip install -q ultralytics

In [ ]:
# 4. Write data.yaml with Colab-local paths
yaml = """path: /content/datasets/crack.yolov8
train: train/images
val:   valid/images
test:  test/images
nc: 1
names: ['crack']
"""
with open('/content/data.yaml', 'w') as f:
    f.write(yaml)
print('data.yaml written')

In [ ]:
# 5. Train — saves best.pt + last.pt to Drive after every epoch
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')   # nano-seg, ~3MB, fast

results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=15,          # early stop if no improvement for 15 epochs
    save=True,
    save_period=1,        # save every epoch to Drive
    project=SAVE_DIR,
    name='run1',
    exist_ok=True,
    device=0,             # GPU
    workers=2,
    seed=42,
)

In [ ]:
# 6. Validate final metrics
best = f'{SAVE_DIR}/run1/weights/best.pt'
model = YOLO(best)
metrics = model.val(data='/content/data.yaml', device=0)
print(f"mAP50:    {metrics.seg.map50:.3f}")
print(f"mAP50-95: {metrics.seg.map:.3f}")
print(f"best.pt -> {best}")

In [ ]:
# 7. Export to ONNX (for FastAPI)
model.export(format='onnx', imgsz=640, simplify=True)
import shutil
shutil.copy(best.replace('.pt', '.onnx'), SAVE_DIR + '/crack_seg.onnx')
print('ONNX saved to Drive')

## Resume after disconnect
```python
model = YOLO(f'{SAVE_DIR}/run1/weights/last.pt')
model.train(resume=True)
```

## Improve accuracy (fine-tune from best)
```python
model = YOLO(f'{SAVE_DIR}/run1/weights/best.pt')
model.train(data='/content/data.yaml', epochs=50, lr0=0.001, project=SAVE_DIR, name='run2')
```